In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import matplotlib.animation as animation
import matplotlib.colors as colors
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128

from IPython.display import HTML
from astropy.stats import sigma_clip
from tesscube import TESSCube
from astropy.time import Time
from tqdm import tqdm

from tess_asteroids import MovingTPF, __version__
from scipy import stats
from matplotlib import patches
from scipy import signal, ndimage
from datetime import datetime

DATADIR = "/Users/jimartin/Work/TESS/tess-asteroid-ml/data"

# We open the track file

In [ ]:
sector = 26
camera = 2
ccd = 2
trackno = 1

In [ ]:
track = pd.read_csv(f"{DATADIR}/pred_asteroid_tracks/sector{sector:04}/camera{camera}/asteroid_xy_{sector}_{camera}_{ccd}_track_{trackno}_bspline_3.txt")
track.rename(columns={"Time (JD)": "jd"}, inplace=True)
X = track["X"].values.copy()
Y = track["Y"].values.copy()
mask = track["Flag"] == "F"
X[mask] = track["Y"][mask].values.copy()
Y[mask] = track["X"][mask].values.copy()
track["X"] = X
track["Y"] = Y
track.head(20)

In [ ]:
track.info()

In [ ]:
# plt.scatter(track.X_fit, track.Y_fit, s=1)
plt.title(f"Asteroid Track in Sector {sector} Camera {camera} CCD {ccd}")
plt.scatter(track.query("Flag == 'P'").X, track.query("Flag == 'P'").Y, s=2, c="tab:red", label="Projected")
plt.scatter(track.query("Flag == 'F'").X, track.query("Flag == 'F'").Y, s=2, c="tab:blue", label="Detected")
plt.xlabel("Column")
plt.ylabel("Row")
plt.xlim(0, 2048)
plt.ylim(0, 2048)
plt.legend()
plt.show()

In [ ]:
ephem = pd.DataFrame({
            "time": track.jd - 2457000,
            "sector": sector,
            "camera": camera,
            "ccd": ccd,
            "column": track.X_fit,
            "row": track.Y_fit,
        })


ephem

In [ ]:
# Initialise
target = MovingTPF("2024 ASDF", ephem)

In [ ]:
target.make_tpf(shape=(41,41), file_name="./data/test.fits")

In [ ]:
target.get_data(shape=(21,21))
target.reshape_data()

In [ ]:
target.background_correction(method="rolling", nframes=31)
target.create_pixel_quality()

In [ ]:
target.create_aperture(method="prf")

In [ ]:
target.cadence_number

In [ ]:
anim_fname = f"./data/tess_{target.target.replace(' ', '')}_s{sector:04}-{camera}-{ccd}_track{trackno:02}_anim_{'prf'}.gif"

target.animate_tpf(cnorm=False, step=2)

In [ ]:
mtpf_dir = f"{DATADIR}/moving_TPFs/sector{sector:04}"

hdul, fname = target._save_tpf(file_name=None, save_loc=mtpf_dir, return_hdu=True)

hdul[0].header.set("DATE", datetime.now().strftime("%Y-%m-%d"), comment="file creation date")
hdul[0].header.set("CREATOR", "tess-asteroids", comment="software used to produce this file")
hdul[0].header.set("VERSION", __version__, comment="software version")

hdul[0].header.set("TSTART", target.time[0], comment="observation start time in TJD of first frame")
hdul[0].header.set("TSTOP", target.time[-1], comment="observation start time in TJD of last frame")
hdul[0].header.set("DATE-OBS", Time(target.time[0], format="btjd", scale="tdb").utc.isot, comment="TSTART as UTC calendar date")
hdul[0].header.set("DATE-END", Time(target.time[-1], format="btjd", scale="tdb").utc.isot, comment="TSTOP as UTC calendar date")

hdul[0].header.set("OBJECT", "2024 Brian", comment="Asteroid name")
hdul[0].header.set("VMAG", "", comment="V magnitude")
hdul[0].header.set("HMAG", "", comment="H absolute magnitude")
hdul[0].header.set("PERIHEL", "", comment="Perihelion (au)")
hdul[0].header.set("ORBECC", "", comment="Orbit eccentricity")
hdul[0].header.set("ORBINC", "", comment="Orbit inclination [deg]")
hdul[0].header.set("RARATE", "", comment='Right ascension rate ["/h]')
hdul[0].header.set("DECRATE", "", comment='Declination rate ["/h]')

In [ ]:
fname

In [ ]:
hdul.writeto(fname, overwrite=True)

# Save/Load with LightKurve

In [ ]:
tpf = lk.read("./data/tess-asteroid_001-s0007-shape25x25-moving_tp.fits", 
              quality_bitmask="none")

In [ ]:
tpf.animate(column="Flux", vmin=0, vmax=12)

In [ ]:
from astropy.io import fits

In [ ]:
hdul = fits.open("./data/tess-asteroid_001-s0007-shape25x25-moving_tp.fits")

In [ ]:
hdul.info()

In [ ]:
hdul[1].columns

In [ ]:
tpf.get_header()

In [ ]:
tpf.plot(frame=10, column="FLUX_BKG")

# Aperture mask

## Using a threshold

In [ ]:
aperture_mask = target._create_threshold_mask(reference_pixel="center")

In [ ]:
target.create_aperture(method="threshold", threshold=3, reference_pixel="center")

In [ ]:
np.all(aperture_mask == target.aperture_mask)

In [ ]:
tap_flux, tap_flux_error = np.array(
    [[np.nansum(x[m]), 
      np.nansum(x[m] ** 2) ** 0.5] 
     for x, m in zip((target.flux - target.bg), target.aperture_mask)]
).T

In [ ]:
plt.figure(figsize=(12,3))
plt.title("Asteroid Light Curve")
plt.errorbar(target.time, tap_flux, yerr=tap_flux_error, fmt=".", c="tab:blue", label="Fitted track")
plt.xlabel("Time [TBJD]")
plt.ylabel("Flux [-e/s]")
plt.ylim(-20, 300)
plt.legend()
plt.show()

In [ ]:
def plot_img_aperture(
    img, 
    aperture_mask=None, 
    cbar=True,
    ax=None,
    corner=[0, 0],
    xy=None,
    title=None,
    vmin=None,
    vmax=None,
):

    if ax is None:
        fig, ax = plt.subplots()
        fig.suptitle(f"Asteroid Tracks")

    extent = (
              corner[1] - 0.5,
              corner[1] + img.shape[1] - 0.5,
              corner[0] - 0.5,
              corner[0] + img.shape[0] - 0.5,
    )
    
    im = ax.imshow(
        img, 
        cmap="viridis",
        vmin=vmin, 
        vmax=vmax,
        rasterized=True,
        origin="lower",
        extent=extent,
    )
    if cbar:
        plt.colorbar(im, location="right", shrink=.8, label="Flux [-e/s]")
    if xy is not None:
        dot = ax.scatter(xy[1], xy[0], marker="o", c="red", alpha=1, s=15)
        
    ax.set_aspect('equal', 'box')
    ax.set_title(title)

    if aperture_mask is not None:
        row, col = np.mgrid[corner[0] : corner[0] + img.shape[0], 
                            corner[1] : corner[1] + img.shape[1]]
        for i, pi in enumerate(row[:, 0]):
            for j, pj in enumerate(col[0, :]):
                if aperture_mask[i, j]:
                    # print("here")
                    rect = patches.Rectangle(
                        xy=(pj - 0.5, pi - 0.5),
                        width=1,
                        height=1,
                        color="tab:red",
                        fill=False,
                        # hatch="//",
                        alpha=0.8,
                    )
                    ax.add_patch(rect)

    ax.set_xlabel("Pixel Column")
    ax.set_ylabel("Pixel Row")

    return ax

def animate_image(cube, 
                  aperture_mask=None, 
                  corner=[0, 0], 
                  track=None,
                  cadenceno=None,
                  time=None,
                  interval=200, 
                  repeat_delay=1000
                 ):
    vlo, lo, mid, hi, vhi = np.nanpercentile(cube, [0.2, 1, 50, 99, 99.8])
    
    fig, ax = plt.subplots()
    fig.suptitle(f"Asteroid Tracks in Sector {sector} Camera {camera} CCD {ccd}")

    if aperture_mask is None:
        aperture_mask = np.repeat([None], len(cube), axis=0)
    else:
        if aperture_mask.shape == cube.shape[1:]:
            aperture_mask = np.repeat([aperture_mask], len(cube), axis=0)
            
    if track is None:
        track = np.repeat([None], len(cube), axis=0)
    if cadenceno is None:
        cadenceno = np.repeat([None], len(cube), axis=0)
    if time is None:
        time = np.repeat([None], len(cube), axis=0) 
        
    if len(corner) == 2:
        corner = np.repeat([corner], len(cube), axis=0)

    nt = 0
    ax = plot_img_aperture(cube[nt], 
                           aperture_mask=aperture_mask[nt], 
                           cbar=True,
                           ax=ax,
                           corner=corner[nt],
                           xy=track[nt],
                           title=f"CAD {cadenceno[nt]} | BTJD {time[nt]:.4f}",
                           vmin=lo,
                           vmax=hi,
                          )

    def animate(nt):
        ax.clear()
        _ = plot_img_aperture(cube[nt], 
                              aperture_mask=aperture_mask[nt], 
                              cbar=False,
                              ax=ax,
                              corner=corner[nt],
                              xy=track[nt],
                              title=f"CAD {cadenceno[nt]} | BTJD {time[nt]:.4f}",
                              vmin=lo,
                              vmax=hi,
                             )
        
        return ()

    plt.close(ax.figure)

    # Create the animation
    ani = animation.FuncAnimation(
        fig, animate, 
        frames=len(cube), 
        interval=interval, 
        blit=True, 
        repeat_delay=repeat_delay, 
        repeat=True,
    )

    return ani
    

In [ ]:
HTML(
    animate_image(target.flux - target.bg, 
                  aperture_mask=target.aperture_mask, 
                  corner=target.corner, 
                  track=target.ephemeris,
                  cadenceno=target.cadence_number,
                  time=target.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)

In [ ]:
# image gradients to identify subtraction artifacts 

nt = 11
fig, ax = plt.subplots(2, 2, figsize=(9,9))

ax[0, 0].set_title("df/dy")
ax[0, 0].imshow(
    np.gradient((target.flux - target.bg)[nt])[0], 
    cmap="viridis",
    # vmin=0, 
    # vmax=0.2,
    origin="lower",
    extent=(target.corner[nt, 1]-0.5, target.corner[nt, 1] +target.shape[1] - 0.5,
            target.corner[nt, 0]-0.5, target.corner[nt, 0] +target.shape[0] - 0.5,
           )
    )
ax[0, 1].set_title("df/dx")
ax[0, 1].imshow(
    np.gradient((target.flux - target.bg)[nt])[1], 
    cmap="viridis",
    # vmin=0, 
    # vmax=0.2,
    origin="lower",
    extent=(target.corner[nt, 1]-0.5, target.corner[nt, 1] +target.shape[1] - 0.5,
            target.corner[nt, 0]-0.5, target.corner[nt, 0] +target.shape[0] - 0.5,
           )
    )

ax[1, 0].set_title("df$^2$/dy$^2$")
ax[1, 0].imshow(
    np.gradient(np.gradient((target.flux - target.bg)[nt])[0])[0], 
    cmap="viridis",
    # vmin=0, 
    # vmax=0.2,
    origin="lower",
    extent=(target.corner[nt, 1]-0.5, target.corner[nt, 1] +target.shape[1] - 0.5,
            target.corner[nt, 0]-0.5, target.corner[nt, 0] +target.shape[0] - 0.5,
           )
    )
ax[1, 1].set_title("df$^2$/dx$^2$")
ax[1, 1].imshow(
    np.gradient(np.gradient((target.flux - target.bg)[nt])[1])[1], 
    cmap="viridis",
    # vmin=0, 
    # vmax=0.2,
    origin="lower",
    extent=(target.corner[nt, 1]-0.5, target.corner[nt, 1] +target.shape[1] - 0.5,
            target.corner[nt, 0]-0.5, target.corner[nt, 0] +target.shape[0] - 0.5,
           )
    )
plt.show()

## Aperture with Ellipse

In [ ]:
from matplotlib.patches import Ellipse
from numpy.polynomial import Polynomial
from astropy.stats import sigma_clip

In [ ]:
target.cube.tstart

In [ ]:
ellip_mask, ellip_params = target._create_ellipse_aperture(
    R=3, 
    smooth=True, 
    return_params=True, 
    plot=True
)

In [ ]:
plt.plot(target.time, ellip_params[:, 2])
plt.plot(target.time, ellip_params[:, 3])

In [ ]:
test, _ = MovingTPF.from_name("1998 YT6", sector=6)
test.get_data(shape=(11, 11))
test.reshape_data()
test.background_correction(method="rolling")

In [ ]:
ellip_mask, params = test._create_ellipse_aperture(return_params=True, R=3)

In [ ]:
plt.plot(test.time, params[:, -1]);

In [ ]:
nt = 20

fig, ax = plt.subplots(1,1)
ax = plot_img_aperture(
    (test.flux - test.bg)[nt], 
    aperture_mask=ellip_mask[nt], 
    ax=ax,
    cbar=True,
    corner=test.corner[nt],
    xy=test.ephemeris[nt],
    title=f"CAD {test.cadence_number[nt]} | BTJD {test.time[nt]:.4f}",
    vmin=0,
    vmax=12,
    )
ax.scatter(params[nt, 0], params[nt, 1], c="cyan", s=10)
ellipse = Ellipse(
    tuple(np.flip(test.ephemeris[nt])),
    # tuple([column_centroid[nt] + test.corner[nt, 1], row_centroid[nt] + test.corner[nt, 0]]), 
    width=params[nt, 2] * 4,
    height=params[nt, 3] * 4,
    angle=params[nt, 4],
    facecolor="none", 
    edgecolor="red",
    lw=2,
)
ax.add_patch(ellipse)

plt.show()

In [ ]:
HTML(
    animate_image(test.flux - test.bg, 
                  aperture_mask=ellip_mask, 
                  corner=test.corner, 
                  track=test.ephemeris,
                  cadenceno=test.cadence_number,
                  time=test.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)

In [ ]:
nt = 20

fig, ax = plt.subplots(1,1)
ax = plot_img_aperture(
    (target.flux - target.bg)[nt], 
    aperture_mask=ellip_mask[nt], 
    ax=ax,
    cbar=True,
    corner=target.corner[nt],
    xy=target.ephemeris[nt],
    title=f"CAD {target.cadence_number[nt]} | BTJD {target.time[nt]:.4f}",
    vmin=0,
    vmax=12,
    )
ax.scatter(ellip_params[nt, 0], ellip_params[nt, 1], c="cyan", s=10)
ellipse = Ellipse(
    tuple(np.flip(target.ephemeris[nt])),
    # tuple([column_centroid[nt] + target.corner[nt, 1], row_centroid[nt] + target.corner[nt, 0]]), 
    width=ellip_params[nt, 2] * 4,
    height=ellip_params[nt, 3] * 4,
    angle=ellip_params[nt, 4],
    facecolor="none", 
    edgecolor="red",
    lw=2,
)
ax.add_patch(ellipse)

plt.show()

In [ ]:
HTML(
    animate_image(target.flux - target.bg, 
                  aperture_mask=ellip_mask, 
                  corner=target.corner, 
                  track=target.ephemeris,
                  cadenceno=target.cadence_number,
                  time=target.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)

# Aperture with LKPRF 

In [ ]:
a = np.random.normal(size=(5, 20, 20))
a /= a.sum(axis=-1).sum(axis=-1)[:, None, None]

In [ ]:
a[0].sum()

In [ ]:
aa = sum(a) / np.sum(a)

In [ ]:
sum(a).shape

In [ ]:
a.sum(axis=-1).sum(axis=-1)

In [ ]:
import lkprf
from scipy.interpolate import UnivariateSpline

In [ ]:
def get_fwhm(dist):
    """
    Estimates the FWHM of a distribution by computing 
    the roots of an interpolated shifted distribution

    Parameters
    ----------
    dist: ndarray
        Array of 1D distribution to find the FWHM

    Returns
    -------
    fwhm: float
        FWHM value
    """

    # find the half max of the distribution to shift it in the y-axis
    half_max = dist.max() / 2

    # do the interpolation
    x = np.linspace(0, len(dist), len(dist))
    spline = UnivariateSpline(x, dist - half_max, s=0)
    # finds roots of the interpolated curve, i.e. the x values where 
    # the distribution is half the max 
    r1, r2 = spline.roots()
    # return the full width
    return r2 - r1

def _prf_aperture(self, threshold=0.01, return_prf=False, plot=False):
    """
    Creates an aperture mask using the Camera/CCD PRF model.

    Parameters
    ----------
    threshold: float
        PRF pixel value at which pixels are cut to select an aperture

    Returns
    -------
    aperture_mask: ndarray
        Boolean 3D mask array with pixels within the PRF.
    """
    # initialize lkPRF object with Cam-CCD
    prf = lkprf.TESSPRF(camera=self.camera, ccd=self.ccd)

    # find the average PRF FWHM
    xprof, yprof = prf[0].sum(axis=1), prf[0].sum(axis=0)
    fwhm = np.mean([get_fwhm(xprof), get_fwhm(yprof)])

    # The asteroid movement from the ephemeris is projected on both axes.
    dcol = np.gradient(self.ephemeris[:, 1])
    drow = np.gradient(self.ephemeris[:, 0])
    proj_speed = np.hypot(dcol, drow)

    # compute eccentricity and elongation from ellipse parameters if computed
    # ellip = 1 - ellip_params[:, 3] / ellip_params[:, 2]
    # elong = ellip_params[:, 2] / ellip_params[:, 3]
    # eccen = np.sqrt(1 - (ellip_params[:, 3] / ellip_params[:, 2]) ** 2)

    # estimate number of PRFs per frame as the speed / PRF size
    # with a minimum of 1 
    # HARDCODED: ntpoints * 2 to make a finer grid of PRFs and have a smoother model
    # this should be more robust
    npoints = (np.round(proj_speed / np.round(fwhm)).astype(int) + 1) * 2

    # find the location of the PRFs along the semi-major axis 
    # first, we find the start and end of a line runs along the asteroid streak
    # HARDCODED: 1/3 of speed, this should be improved to make it more robust.
    # It should be closer to 2 as the speed is dcol / 30 min and integration
    # time of FFI is 30 min, then adding/subtracting half dcol should give the
    # right estimation, but it seems to be overestimated.
    col_line = np.array([self.ephemeris[:, 1] - dcol / 3, self.ephemeris[:, 1] + dcol / 3]).T
    row_line = np.array([self.ephemeris[:, 0] - drow / 3, self.ephemeris[:, 0] + drow / 3]).T
    
    streak_points = []
    prf_aster = []
    
    # N points evenly spaced along the row/column line
    # and evaluate PRF at each of those points all on a frame
    for tt in range(len(self.time)):
        if npoints[tt] > 1:
            locations = (
                np.array([
                    np.linspace(row_line[tt, 0], row_line[tt, 1], npoints[tt]),
                    np.linspace(col_line[tt, 0], col_line[tt, 1], npoints[tt])
                ]
                        ).T
            )
        else:
            # we make sure each element of streak_points is a 2D array
            # even when only one point of the PRF needs is evaluated
            locations = (np.atleast_2d(self.ephemeris[tt]))

        streak_points.append(locations)
        
        # evaluate the PRF from lkprf at every position in every frame
        # frames have multiple positions depending on the the size of the streak
        prf_nt = prf.evaluate(targets=[tuple(x) for x in locations], 
                              origin=tuple(self.corner[tt]), 
                              shape=self.shape)
        # renormalize the motion-blurred PRF
        prf_aster.append(prf_nt.sum(axis=0) / prf_nt.sum())

    # put the PRF in an array and make aperture mask with threshold
    prf_aster = np.array(prf_aster)
    aperture_mask = prf_aster >= threshold

    if plot:
        # plot the 2D image, ellipse, and the line along the streak with the PRF positions
        nt = 11

        fig, ax = plt.subplots(1,1)
        ax = plot_img_aperture(
            self.corr_flux[nt], 
            aperture_mask=aperture_mask[nt], 
            ax=ax,
            cbar=True,
            corner=self.corner[nt],
            xy=self.ephemeris[nt],
            title=f"CAD {self.cadence_number[nt]} | BTJD {self.time[nt]:.4f}",
            vmin=0,
            vmax=12,
            )
        
        ax.plot(col_line[nt], row_line[nt], c="w")
        ax.scatter(streak_points[nt][:, 1], streak_points[nt][:, 0], c="w")
        
        plt.show()

    if return_prf:
        return aperture_mask, prf_aster
    else:
        return aperture_mask

In [ ]:
prf_mask, prf_aster = _prf_aperture(target, return_prf=True, plot=True)

In [ ]:
prf_aster.shape

In [ ]:
FLFRCSAP = [np.sum(prf_aster[k, prf_mask[k]]) for k in range(len(prf_mask))]
# FLFRCSAP

In [ ]:
# movie with the PRF model per frame
HTML(
    animate_image(prf_aster, 
                  aperture_mask=prf_mask, 
                  corner=target.corner, 
                  track=target.ephemeris,
                  cadenceno=target.cadence_number,
                  time=target.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)

## NN Predicted mask as aperture

In [ ]:
import os

In [ ]:
target.cadence_number

In [ ]:
target_cadno

In [ ]:
def _nn_aperture(self, file_path=None, prob_cut=0.5, return_pred=False):
    """
    Creates an aperture using the predicted mask from the NN model.
    It needs the NN output file with the sparse cube of predictions

    Parameters
    ----------
    file_path: string
        Path to file with NN output. File needs to contain the sparse cube
        with NN prediction, values, times, and cadence number.

    Returns
    -------
    aperture_mask: ndarray
        Boolean 3D mask array from the NN prediction cube.
    """

    if file_path is None or not os.path.isfile(file_path):
        raise ValueError("Please provide a valid path to file.")

    pred_array = np.load(file_path)

    if not np.isin(['pred_indices', 'pred_vals', 'time', 'diff'], list(pred_array.keys())).any():
        raise ValueError("One of the arrays is missing from the NPZ file. Need all of"
                         "['pred_indices', 'pred_vals', 'time', 'diff'].")
    
    # compute cadence number from time arrays
    # in the future FFI pred cube will have the cadence number
    dt = np.nanmedian(np.gradient(pred_array["time"]))
    ffi_cadno = np.round((pred_array["time"] - pred_array["time"].min()) / dt).astype(int)

    target_cadno = np.round((target.time - pred_array["time"].min() + 2457000) / dt).astype(int)

    # row/column arrays needed for cube sparse indices
    ROW, COL = np.arange(2092), np.arange(2092)

    nn_pred = np.zeros_like(target.flux).astype(float)

    for kk, cn in enumerate(target_cadno):
        idx_mask = cn == ffi_cadno[pred_array["pred_indices"][0]]
    
        rr, cc = np.mgrid[target.corner[kk, 0] : target.corner[kk, 0] + target.shape[0],
                          target.corner[kk, 1] : target.corner[kk, 1] + target.shape[1]]
    
        frame_mask = np.zeros_like(rr).astype(float)
        pred_R = ROW[pred_array["pred_indices"][1, idx_mask]]
        pred_C = COL[pred_array["pred_indices"][2, idx_mask]]
        pred_V = pred_array["pred_vals"][idx_mask]
        
        for r, c, v in zip(pred_R, pred_C, pred_V):
            # print(r, c, v)
            frame_mask[(rr == r) & (cc == c)] = v
        nn_pred[kk] = frame_mask

    aperture_mask = nn_pred >= prob_cut
    
    if return_pred:
        return aperture_mask, nn_pred
    else:
        return aperture_mask

In [ ]:
nn_mask2, nn_pred = _nn_aperture(target, 
             file_path=f"/Users/jimartin/Work/TESS/data/nn_predict_ffi/asteroid_{sector}_{camera}_{ccd}.npz",
             return_pred=True)

In [ ]:
pred_array = np.load(f"/Users/jimartin/Work/TESS/data/nn_predict_ffi/asteroid_{sector}_{camera}_{ccd}.npz")
pred_array

In [ ]:
np.isin(['pred_indices', 'pred_vals', 'time'], list(pred_array.keys()))

In [ ]:
list(pred_array.keys())

In [ ]:
pred_array["time"]

In [ ]:
pred_array["pred_indices"]

In [ ]:
pred_array["pred_vals"].shape

In [ ]:
pred_array["diff"].shape

In [ ]:
plt.imshow(pred_array["diff"], origin="lower")
plt.show()

In [ ]:
ffi_cadno = np.round((pred_array["time"] - pred_array["time"].min()) / np.nanmedian(np.gradient(pred_array["time"]))).astype(int)

target_cadno = np.round((target.time - pred_array["time"].min() + 2457000) / np.nanmedian(np.gradient(pred_array["time"]))).astype(int)
ffi_cadno, target_cadno

In [ ]:
ROW, COL = np.arange(2092), np.arange(2092)
ROW

In [ ]:
pred_array["pred_indices"]

In [ ]:
target.cadence_number

In [ ]:
nn_mask = np.zeros_like(target.flux).astype(float)

for kk, cn in enumerate(target_cadno):
    idx_mask = cn == ffi_cadno[pred_array["pred_indices"][0]]

    rr, cc = np.mgrid[target.corner[kk, 0] : target.corner[kk, 0] + target.shape[0],
                      target.corner[kk, 1] : target.corner[kk, 1] + target.shape[1]]

    frame_mask = np.zeros_like(rr).astype(float)
    pred_R = ROW[pred_array["pred_indices"][1, idx_mask]]
    pred_C = COL[pred_array["pred_indices"][2, idx_mask]]
    pred_V = pred_array["pred_vals"][idx_mask]
    
    for r, c, v in zip(pred_R, pred_C, pred_V):
        # print(r, c, v)
        frame_mask[(rr == r) & (cc == c)] = v
    nn_mask[kk] = frame_mask
    # break

    # break

In [ ]:
# plot the 2D image, ellipse, and the line along the streak with the PRF positions
nt = 0

fig, ax = plt.subplots(1,1)
ax = plot_img_aperture(
    nn_mask[nt],
    # (target.flux - target.bg)[nt], 
    # aperture_mask=nn_mask[nt], 
    ax=ax,
    cbar=True,
    corner=target.corner[nt],
    # xy=target.ephemeris[nt],
    title=f"CAD {target.cadence_number[nt]} | BTJD {target.time[nt]:.4f}",
    vmin=0,
    vmax=1,
    )

plt.show()

In [ ]:
HTML(
    animate_image(
                  nn_pred,
                  # target.flux - target.bg, 
                  aperture_mask=nn_mask2, 
                  corner=target.corner, 
                  track=target.ephemeris,
                  cadenceno=target.cadence_number,
                  time=target.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)

In [ ]:
test, _ = MovingTPF.from_name("1998 YT6", sector=6)
test.get_data(shape=(11, 11))
test.reshape_data()
test.background_correction(method="rolling")

In [ ]:
ellip_mask, ell_params = test._create_ellipse_aperture(R=2, return_params=True, plot=True)

In [ ]:
assert np.isfinite(ell_params).all()
assert np.mean(ell_params[:, 0] - test.ephemeris[:, 1]) < 0.2
assert np.mean(ell_params[:, 1] - test.ephemeris[:, 0]) < 0.1
assert ((ell_params[:, 4] >= 0) & (ell_params[:, 4] <=  360)).all()

In [ ]:
plt.plot(test.time, ell_params[:, 2])
plt.plot(test.time, ell_params[:, 3])

In [ ]:
plt.plot(test.time, ell_params[:, 4])

In [ ]:
np.median(ellip_mask.reshape((test.time.shape[0], -1)).sum(axis=1))

In [ ]:
assert ellip_mask.shape == test.flux.shape
assert np.median(ellip_mask.reshape((test.time.shape[0], -1)).sum(axis=1)) == 13

assert ell_params.shape == (test.time.shape[0], 5)

In [ ]:
HTML(
    animate_image(
                  test.corr_flux,
                  aperture_mask=ellip_mask, 
                  corner=test.corner, 
                  track=test.ephemeris,
                  cadenceno=test.cadence_number,
                  time=test.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)